# 02 — Beam Metrics & FITS Export

Demonstrates all standard beam characterisation quantities in `beamlab21.metrics`,  
aperture efficiency, and exporting a beam cube to FITS.

**No external data needed** — a synthetic Gaussian beam is built inline.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from beamlab21.metrics import (
    aperture_efficiency,
    beam_solid_angle,
    beam_summary,
    directivity,
    hpbw,
    main_lobe_efficiency,
)
from beamlab21.models import twoD_Gaussian

## 1 · Synthesise a beam

In [ ]:
N_pix  = 301
xy_max = 40.0
x = np.linspace(-xy_max, xy_max, N_pix)
y = np.linspace(-xy_max, xy_max, N_pix)

# [amp, sigx, sigy, xo, yo, tilt_deg]
TRUE_GAUSS = [1.0, 9.0, 8.0, 0.5, -0.3, 0.0]
sigx, sigy = TRUE_GAUSS[1], TRUE_GAUSS[2]
xo,   yo   = TRUE_GAUSS[3], TRUE_GAUSS[4]
freq_mhz   = 400.0

beam = twoD_Gaussian(x, y, TRUE_GAUSS)
beam = beam / beam.max()   # normalise to peak = 1

print(f"Grid   : {N_pix}×{N_pix}  (±{xy_max}°)")
print(f"Beam   : sigx={sigx}°  sigy={sigy}°  xo={xo}°  yo={yo}°")

## 2 · Individual metrics

In [ ]:
hx, hy = hpbw(sigx, sigy)
sa     = beam_solid_angle(beam, x, y)
mle    = main_lobe_efficiency(beam, x, y, xo, yo, sigx, sigy, fac=1.5)
div    = directivity(beam)

print(f"HPBW x          : {hx:.3f} °")
print(f"HPBW y          : {hy:.3f} °")
print(f"Beam solid angle: {sa:.4f} deg²")
print(f"Main-lobe eff.  : {mle:.4f}  ({mle*100:.1f} %)")
print(f"Directivity     : {div:.2f}")

In [ ]:
# All at once via beam_summary
summary = beam_summary(sigx, sigy, beam, x, y, xo=xo, yo=yo, fac=1.5)
print("beam_summary output:")
for k, v in summary.items():
    print(f"  {k:<22}: {v}")

## 3 · Aperture efficiency

Computes η_ap = λ² / (4π · Ω_A · A_geom) from the Gaussian beam width.

In [ ]:
DISH_M = 6.0   # dish diameter in metres

ae = aperture_efficiency(freq_mhz=freq_mhz, sigx=sigx, sigy=sigy,
                          dish_diameter_m=DISH_M)
print(f"Aperture efficiency  : {ae['eta_ap']:.4f}  ({ae['eta_ap']*100:.1f} %)")
print(f"Effective area       : {ae['A_eff_m2']:.3f} m²")
print(f"Geometric area       : {ae['A_geom_m2']:.3f} m²")

In [ ]:
# How aperture efficiency varies with frequency (fixed Gaussian width)
freqs = np.linspace(300, 1000, 100)
etas  = [aperture_efficiency(f, sigx, sigy, DISH_M)["eta_ap"] for f in freqs]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(freqs, etas, color="steelblue")
ax.axvline(freq_mhz, color="r", linestyle="--", label=f"{freq_mhz:.0f} MHz")
ax.set_xlabel("Frequency [MHz]")
ax.set_ylabel("η_ap")
ax.set_title(f"Aperture efficiency vs frequency  (D={DISH_M} m, σ fixed)")
ax.legend()
plt.tight_layout()
plt.show()

## 4 · Export to FITS

In [ ]:
from beamlab21.io import save_fits

freq_arr_mhz = np.array([400.0, 500.0, 600.0])

# Synthesise a 3-channel cube (beam narrows with frequency)
cube = np.stack(
    [twoD_Gaussian(x, y, [1.0, sigx * (freq_mhz / f)**1.0, sigy * (freq_mhz / f)**1.0, xo, yo, 0.0])
     for f in freq_arr_mhz],
    axis=0,
)  # shape (3, N_pix, N_pix)

_tmp = tempfile.NamedTemporaryFile(suffix=".fits", delete=False)
_tmp.close()
fits_path = _tmp.name

save_fits(fits_path, x, y, freq_arr_mhz, cube, overwrite=True)
print(f"FITS written → {fits_path}")

# Quick inspection with astropy
try:
    from astropy.io import fits as apfits
    with apfits.open(fits_path) as hdul:
        hdr = hdul[0].header
        print(f"FITS shape : {hdul[0].data.shape}")
        print(f"NAXIS      : {hdr['NAXIS']}")
        print(f"CTYPE1     : {hdr.get('CTYPE1','—')}")
        print(f"CTYPE3     : {hdr.get('CTYPE3','—')}")
except ImportError:
    print("(astropy not installed — skipping header inspection)")

Path(fits_path).unlink(missing_ok=True)

## 5 · Visual summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Beam map with HPBW ellipse
ax = axes[0]
ax.pcolormesh(x, y, beam, cmap="inferno", vmin=0, vmax=1)
from matplotlib.patches import Ellipse
ell = Ellipse((xo, yo), width=hx, height=hy, fill=False, edgecolor="cyan", linewidth=1.5,
              label=f"HPBW ({hx:.1f}°×{hy:.1f}°)")
ax.add_patch(ell)
ax.legend(fontsize=8)
ax.set_title("Beam map + HPBW ellipse")
ax.set_xlabel("x [deg]"); ax.set_ylabel("y [deg]")

# 1D cuts
ax = axes[1]
mid = N_pix // 2
ax.plot(x, beam[mid, :], label="EW cut")
ax.plot(y, beam[:, mid], label="NS cut", linestyle="--")
ax.axhline(0.5, color="k", linewidth=0.7, linestyle=":")
ax.set_xlabel("Angle [deg]")
ax.set_ylabel("Normalised amplitude")
ax.set_title("1D cuts through beam centre")
ax.legend()

# Metrics bar chart
ax = axes[2]
names  = ["HPBW_x\n[deg]", "HPBW_y\n[deg]", "Solid angle\n[deg²]",
          "MLE\n[%]", "Directivity"]
values = [hx, hy, sa, mle * 100, div]
bars = ax.bar(names, values, color=["steelblue"] * 5, edgecolor="k", linewidth=0.5)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
            f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_title("Beam metrics summary")

plt.tight_layout()
plt.show()